# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [2]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from anthropic import Anthropic

In [3]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('CLAUDE_API_KEY')

if api_key and api_key.startswith('sk-ant-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'claude-sonnet-5-5'
claude = Anthropic(api_key=api_key)

API key looks good so far


In [4]:
links = fetch_website_links("https://edwarddonner.com")
links

['#wp--skip-link--target',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/2026/11/11/ai-sdlc-the-ai-native-software-engineer-in-the-agentic-era/',
 'https://edwarddonner.com/2026/11/11/ai-sdlc-the-ai-native-software-engineer-in-the-agentic-era/',
 'https://edwarddonner.com/2026/10/14/ai-fde-the-forward-deployed-engineer-agentic-ai-architect/',
 'https://edwarddonner.com/2026/10/14/ai-fde-the-forward

## First step: Have GPT-6 Luna figure out which links are relevant

### Use a call to GPT-6 Luna to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [5]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [6]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [7]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

#wp--skip-link--target
https://edwarddonner.com/avatar/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/avatar/
https://edwarddonner.com/2026/11/11/ai-sdlc-the-ai-native-software-engineer-in-the-agentic-era/
https://edwarddo

In [36]:
import json

def parse_json(text):
    start = text.index("{")
    obj,_ = json.JSONDecoder().raw_decode(text[start:])
    return obj

In [37]:
def select_relevant_links(url):
    response = claude.messages.create(
        model="claude-haiku-4-5",
        messages=[
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        system=link_system_prompt,
        max_tokens=4096
    )
    result = response.content[0].text
    links = parse_json(result)
    return links
    

In [38]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'home page', 'url': 'https://edwarddonner.com/'},
  {'type': 'curriculum/services',
   'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'blog/posts', 'url': 'https://edwarddonner.com/posts/'},
  {'type': 'portfolio/projects', 'url': 'https://edwarddonner.com/avatar/'},
  {'type': 'portfolio/projects',
   'url': 'https://edwarddonner.com/proficient/'},
  {'type': 'portfolio/projects',
   'url': 'https://edwarddonner.com/connect-four/'},
  {'type': 'portfolio/projects', 'url': 'https://edwarddonner.com/outsmart/'},
  {'type': 'social media', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'social media', 'url': 'https://twitter.com/edwarddonner'}]}

In [ ]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        reasoning_effort="none",
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [ ]:
select_relevant_links("https://edwarddonner.com")

In [39]:
select_relevant_links("https://huggingface.co")

{'links': [{'type': 'about page', 'url': 'https://huggingface.co/'},
  {'type': 'company info', 'url': 'https://huggingface.co/brand'},
  {'type': 'models page', 'url': 'https://huggingface.co/models'},
  {'type': 'datasets page', 'url': 'https://huggingface.co/datasets'},
  {'type': 'spaces page', 'url': 'https://huggingface.co/spaces'},
  {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'documentation', 'url': 'https://huggingface.co/docs'},
  {'type': 'blog', 'url': 'https://huggingface.co/blog'},
  {'type': 'learning resources', 'url': 'https://huggingface.co/learn'},
  {'type': 'community chat', 'url': 'https://huggingface.co/chat'},
  {'type': 'discord community', 'url': 'https://huggingface.co/join/discord'},
  {'type': 'discussion forum', 'url': 'https://discuss.huggingface.co/'},
  {'type': 'github', 'url': 'https://github.com/huggingface'},
  {'type': 'careers', 'url': 'htt

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-6 Luna

In [40]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [41]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Search models, datasets, users...
Ctrl K
Models
Datasets
Spaces
Buckets
Docs
Enterprise
Pricing
Website
Tasks
HuggingChat
Collections
Languages
Organizations
Community
Blog
Posts
Daily Papers
Hardware
Learn
Discord
Forum
GitHub
Solutions
Team & Enterprise
Hugging Face PRO
Enterprise Support
Inference Providers
Inference Endpoints
Storage Buckets
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
thinkingmachines/Inkling-2-Mini
Updated
about 21 hours ago
•
23.5k
•
4.62k
unsloth/Inkling-2-Mini-GGUF
Updated
about 15 hours ago
•
806k
•
826
inworld/voice-isolator
Updated
9 days ago
•
601
tencent/Hy4-flash
Updated
3 days ago
•
6.62k
•
597
apple/FastFrames-L
Updated
3 days ago
•
521
Browse 2M+ models
Spaces
Running
on
CPU Upgrade
819
The Ulti

In [49]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [42]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [43]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nSearch models, datasets, users...\nCtrl K\nModels\nDatasets\nSpaces\nBuckets\nDocs\nEnterprise\nPricing\nWebsite\nTasks\nHuggingChat\nCollections\nLanguages\nOrganizations\nCommunity\nBlog\nPosts\nDaily Papers\nHardware\nLearn\nDiscord\nForum\nGitHub\nSolutions\nTeam & Enterprise\nHugging Face PRO\nEnterprise Support\nInference Providers\nInference Endpoints\nStorage Buckets\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nthinkingmachines/Inkling-2-Mini\nUpdated\nabout 21 hours ago\n•\n23.5k\n•\n4.6

In [50]:
def create_brochure(company_name, url):
    response = claude.messages.create(
        model="claude-sonnet-5-5",
        system=brochure_system_prompt,
        messages=[
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
        max_tokens = 2048
    )
    result = response.content[0].text
    display(Markdown(result))

In [51]:
create_brochure("HuggingFace", "https://huggingface.co")

# Hugging Face: The AI Community Building the Future

*The platform where the machine learning community collaborates on models, datasets, and applications.*

---

## Who We Are

Hugging Face is the home of machine learning. It is a collaboration platform where researchers, developers, and companies create, discover, and share AI. The platform hosts unlimited public models, datasets, and applications, and brings the ML community together in one place.

## The Platform at a Glance

| Resource | Scale |
|---|---|
| **Models** | 2M+ |
| **Datasets** | 500k+ |
| **Spaces (AI apps)** | 1M+ |

- **Models:** Trending models come from organizations such as Thinking Machines, Tencent, Apple, and Unsloth, as well as independent contributors.
- **Datasets:** Contributors include NVIDIA and Google, with datasets for cybersecurity, 3D and simulation assets, pixel art, and more.
- **Spaces:** Interactive demos and apps run directly in the browser. Examples include text-to-image and video generation, a hands-on guide to training Mixture-of-Experts models, and a leaderboard comparing coding-agent harnesses.

## What We Offer

- **Hub:** Host and collaborate on models, datasets, and Spaces.
- **Inference Providers and Inference Endpoints:** Run models in production.
- **Storage Buckets:** Storage for ML assets.
- **HuggingChat:** An open chat experience built on community models.
- **Hugging Face PRO:** Upgraded features for individual users.
- **Team & Enterprise Plans:** Collaboration, security, and support for organizations, plus Enterprise Support.
- **Learning and Community Resources:** Docs, Learn courses, Daily Papers, a Blog, Posts, Collections, a Forum, Discord, and GitHub.

## Customers and Community

Hugging Face serves a wide mix of users:

- **Independent developers and researchers** who share and discover open models and data.
- **Leading technology companies** such as Apple, NVIDIA, Google, Tencent, and Thinking Machines, which publish on the platform.
- **Teams and enterprises** that use the Team & Enterprise plans to build with AI collaboratively and securely.

## Culture

The site points to a culture built on:

- **Openness:** Public sharing of models, datasets, and apps.
- **Collaboration:** Community-driven development through organizations, discussions, and shared Spaces.
- **Education:** Hands-on guides, leaderboards, and learning resources.
- **Pace:** Models and datasets on the platform are updated daily or even hourly.

## Careers

The pages we reviewed don't include specific job listings or hiring details. Prospective recruits who are interested in working on open, community-driven AI can explore the company through its community channels (Discord, Forum, GitHub, Blog) and check the official website for current openings.

## Why Hugging Face?

- **For builders:** A huge library of open models, datasets, and apps to build on.
- **For enterprises:** Secure team plans, dedicated support, and production-ready inference.
- **For investors:** A central hub of the open AI ecosystem, with fast-growing usage and participation from major technology players.
- **For recruits:** A mission-driven environment at the center of the AI community.

---

**Get started:** Explore AI Apps · Browse 2M+ Models · Sign Up at huggingface.co

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [69]:
def stream_brochure(company_name, url):
    with claude.messages.stream(
        model="claude-sonnet-5-5",
        system=brochure_system_prompt,
        messages=[
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
        max_tokens = 4096
    ) as stream:
        response = ""
        display_handle = display(Markdown(""), display_id=True)
        for chunk in stream.text_stream:
            response += chunk
            update_display(Markdown(response), display_id=display_handle.display_id)
    return response

In [70]:
stream_brochure("HuggingFace", "https://huggingface.co")

# Hugging Face: The AI Community Building the Future

**Hugging Face is the platform where the machine learning community collaborates on models, datasets, and applications.** It is often described as "The Home of Machine Learning": a place to create, discover, and collaborate on ML.

---

## What We Offer

**A collaboration platform at massive scale**
- **2M+ models** across text, image, video, audio, and more
- **500k+ datasets** for training and evaluation
- **1M+ applications (Spaces)** that anyone can run and share in the browser
- Hosting and collaboration on unlimited public models, datasets, and apps

**Products and solutions**
- **Spaces:** Interactive ML demos and apps, including agent- and MCP-enabled apps
- **HuggingChat:** Chat interface for open models
- **Inference Providers and Inference Endpoints:** Deploy and run models in production
- **Storage Buckets:** Storage for ML assets
- **Hugging Face PRO:** Upgraded features for individual users
- **Team & Enterprise plans:** Collaboration and security features for organizations, plus Enterprise Support

**Learning and community resources**
- Docs, Learn courses, Blog, Daily Papers, Collections
- Community channels: Forum, Discord, and GitHub

---

## What's Happening on the Platform

The landing page shows a vibrant, fast-moving ecosystem. Trending items include:

- **Models:** thinkingmachines/Inkling-2-Mini, unsloth/Inkling-2-Mini-GGUF, tencent/Hy4-flash, apple/FastFrames-L, inworld/voice-isolator
- **Spaces:** "The Ultimate MoE Training Handbook" (a hands-on guide to training Mixture-of-Experts models), a Harness Leaderboard for coding agents, text-to-image and video generation demos
- **Datasets:** Contributions from organizations such as NVIDIA and Google, covering areas like simulation-ready assets and cybersecurity

---

## Who Uses Hugging Face

- **Researchers and independent developers** sharing and iterating on open models
- **Major technology organizations** publishing on the platform, including NVIDIA, Google, Apple, and Tencent
- **Startups and AI labs**, such as Thinking Machines and Unsloth
- **Teams and enterprises** using paid plans for secure collaboration and production inference

---

## Culture

Hugging Face is built around **open collaboration**. Its identity is centered on the community: sharing models, datasets, and demos publicly, discussing research through Daily Papers and forums, and building in the open on GitHub and Discord.

---

## Careers

The site has a dedicated careers page ("Current Openings"), but the specific roles were not included in the content provided. Prospective recruits should visit the Hugging Face careers page directly to see current openings. Candidates who are active in the community, whether through contributing models, building Spaces, or joining discussions, will be familiar with the culture.

---

## Get Started

- **Explore AI Apps** or **browse 2M+ models** on the homepage
- **Sign up** for free to host and collaborate on public models, datasets, and Spaces
- Explore **Team & Enterprise** plans for organizational needs
- Join the community on **Discord**, the **Forum**, or **GitHub**

'# Hugging Face: The AI Community Building the Future\n\n**Hugging Face is the platform where the machine learning community collaborates on models, datasets, and applications.** It is often described as "The Home of Machine Learning": a place to create, discover, and collaborate on ML.\n\n---\n\n## What We Offer\n\n**A collaboration platform at massive scale**\n- **2M+ models** across text, image, video, audio, and more\n- **500k+ datasets** for training and evaluation\n- **1M+ applications (Spaces)** that anyone can run and share in the browser\n- Hosting and collaboration on unlimited public models, datasets, and apps\n\n**Products and solutions**\n- **Spaces:** Interactive ML demos and apps, including agent- and MCP-enabled apps\n- **HuggingChat:** Chat interface for open models\n- **Inference Providers and Inference Endpoints:** Deploy and run models in production\n- **Storage Buckets:** Storage for ML assets\n- **Hugging Face PRO:** Upgraded features for individual users\n- **Tea

In [71]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

APIConnectionError: Connection error.

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>